<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/0812_ETRIDATA_segformer_%EC%B0%A8%EC%84%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

세그멘테이션으로 차선 인식 데이터를 etri_ai에서 가지고 옴

In [ ]:
# 1단계: 환경 설정 및 데이터 전처리 (전체 데이터 사용)
# JSON 라벨 파일을 마스크 이미지로 변환

# 필요한 라이브러리 설치
!pip install -q opencv-python numpy

# 기본 라이브러리 임포트
import os
import json
import numpy as np
import cv2

print("라이브러리 로드 완료!")

# 압축 파일 해제
print("압축 파일 해제 중...")

# MonoCameraSemanticSegmentation.zip 해제
if os.path.exists('/content/sample_data/MonoCameraSemanticSegmentation.zip'):
    !cd /content/sample_data && unzip -q MonoCameraSemanticSegmentation.zip
    print("✅ MonoCameraSemanticSegmentation.zip 해제 완료!")

# labels.zip 해제
if os.path.exists('/content/sample_data/labels.zip'):
    !cd /content/sample_data && unzip -q labels.zip
    print("✅ labels.zip 해제 완료!")

# 🔍 실제 폴더 구조 확인
print("\n🔍 실제 폴더 구조 확인:")
print("sample_data 폴더 내용:")
!ls -la /content/sample_data/

# 🎯 실제 이미지 폴더 찾기
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',  # 바로 sample_data 아래
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]

IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        print(f"✅ 이미지 폴더 찾음: {path}")
        # 몇 개 파일이 있는지 확인
        png_files = [f for f in os.listdir(path) if f.endswith('.png')]
        print(f"   📁 PNG 파일 수: {len(png_files)}개")
        break

if IMAGE_DIR is None:
    print("❌ 이미지 폴더를 찾을 수 없습니다!")
    print("🔍 sample_data 하위 폴더 전체 탐색:")
    for item in os.listdir('/content/sample_data/'):
        item_path = os.path.join('/content/sample_data/', item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                files = os.listdir(item_path)
                png_count = len([f for f in files if f.endswith('.png')])
                if png_count > 0:
                    print(f"     🖼️ PNG 파일 {png_count}개 발견!")
                    IMAGE_DIR = item_path
            except:
                pass

# 경로 설정
LABEL_DIR = '/content/sample_data/labels'
MASK_DIR = '/content/sample_data/masks_final'

class_map = {
    "background": 0,
    "lane": 1,
}

print("경로 설정 완료!")
print(f"이미지 폴더: {IMAGE_DIR}")
print(f"라벨 폴더: {LABEL_DIR}")
print(f"마스크 저장 폴더: {MASK_DIR}")

# 전체 데이터 현황 파악
print("\n📊 전체 데이터 현황 파악 중...")

if os.path.exists(IMAGE_DIR):
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    daeduk_images = [f for f in all_images if f.startswith('Daeduk')]
    sangam_images = [f for f in all_images if f.startswith('SangamDMC')]

    print(f"🖼️ 이미지 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_images)}개")
    print(f"  - SangamDMC: {len(sangam_images)}개")
    print(f"  - 총 이미지: {len(all_images)}개")

if os.path.exists(LABEL_DIR):
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]
    daeduk_labels = [f for f in all_labels if f.startswith('Daeduk')]
    sangam_labels = [f for f in all_labels if f.startswith('SangamDMC')]

    print(f"🏷️ 라벨 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_labels)}개")
    print(f"  - SangamDMC: {len(sangam_labels)}개")
    print(f"  - 총 라벨: {len(all_labels)}개")

# 매칭 가능한 데이터 분석
print(f"\n🎯 매칭 분석:")

# 변수 초기화
matched_basenames = []
daeduk_matched = []
sangam_matched = []

if IMAGE_DIR and os.path.exists(IMAGE_DIR) and os.path.exists(LABEL_DIR) and len(all_images) > 0 and len(all_labels) > 0:
    # 이미지 베이스명들
    image_basenames = [f.replace('_leftImg8bit.png', '') for f in all_images]
    # 라벨 베이스명들
    label_basenames = [f.replace('_gtFine_polygons.json', '') for f in all_labels]

    # 매칭되는 것들 찾기
    matched_basenames = list(set(image_basenames) & set(label_basenames))

    daeduk_matched = [name for name in matched_basenames if name.startswith('Daeduk')]
    sangam_matched = [name for name in matched_basenames if name.startswith('SangamDMC')]

    print(f"  - Daeduk 매칭: {len(daeduk_matched)}개")
    print(f"  - SangamDMC 매칭: {len(sangam_matched)}개")
    print(f"  - 총 사용가능: {len(matched_basenames)}개")
else:
    if not IMAGE_DIR or not os.path.exists(IMAGE_DIR):
        print("❌ 이미지 폴더를 찾을 수 없습니다!")
    elif not os.path.exists(LABEL_DIR):
        print("❌ 라벨 폴더를 찾을 수 없습니다!")
    elif len(all_images) == 0:
        print("❌ 이미지 파일이 없습니다!")
    elif len(all_labels) == 0:
        print("❌ 라벨 파일이 없습니다!")

print(f"🎯 처리할 총 데이터: {len(matched_basenames)}개")

# 마스크 이미지 생성
print("\n🚀 전체 데이터로 마스크 이미지 생성 시작...")
os.makedirs(MASK_DIR, exist_ok=True)

if len(matched_basenames) == 0:
    print("❌ 처리할 데이터가 없습니다! 경로를 확인해주세요.")
    print("\n✨ 1단계 완료 (데이터 없음)")
else:
    processed_count = 0
    error_count = 0

    # 매칭된 데이터만 처리
    for i, base_name in enumerate(matched_basenames):
        # 파일 경로 구성
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: 이미지 로드 실패 ({image_file})")
            error_count += 1
            continue

        # 이미지 크기 가져오기
        height, width, _ = image.shape

        # JSON 라벨 파일 읽기
        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: JSON 파일 읽기 실패 ({json_file})")
            error_count += 1
            continue

        # 빈 마스크 생성
        mask = np.zeros((height, width), dtype=np.uint8)

        # 차선 폴리곤을 마스크에 그리기
        lane_count = 0
        for obj in data['objects']:
            label = obj['label']
            points = np.array(obj['polygon'], dtype=np.int32)

            # 🎯 실제 데이터셋에서 사용하는 차선 관련 라벨들
            lane_labels = [
                'whdot',           # 흰색 점선 (가장 많음)
                'yedot',           # 노란색 점선
                'blsol',           # 파란색 실선
                'yesol',           # 노란색 실선
                'general road mark' # 일반 도로 표시
            ]

            if label in lane_labels:
                cv2.fillPoly(mask, [points], color=class_map['lane'])
                lane_count += 1

        # 마스크 저장
        mask_file_name = image_file.replace('.png', '_mask.png')
        mask_save_path = os.path.join(MASK_DIR, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        area_type = "🟦 Daeduk" if base_name.startswith('Daeduk') else "🟩 SangamDMC"
        print(f"  ✅ {i+1}/{len(matched_basenames)} {area_type}: {mask_file_name} (차선 {lane_count}개)")

    print(f"\n🎉 전체 데이터 마스크 생성 완료!")
    print(f"✅ 성공: {processed_count}개 (Daeduk + SangamDMC)")
    print(f"❌ 실패: {error_count}개")
    print(f"📁 저장 위치: {MASK_DIR}")

    # 생성된 파일 확인
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_mask.png')]
    print(f"📊 생성된 마스크 파일 수: {len(mask_files)}개")

    print("\n✨ 1단계 완료! 다음 단계로 진행하세요.")

현재 상황:

기존: 2클래스 (배경 + 차선)
새로 만든 것: 8클래스 (배경 + 7개 목표)

🔄 새로운 실행 순서:

다중 클래스 버전 실행 ← 지금 준 코드

기존 1단계 대신에 실행
8클래스 마스크 생성


2단계 실행 (기존과 동일)

데이터셋 생성 및 분할


3단계 수정 실행

모델 설정 (8클래스용으로)


4-5단계 실행

훈련 설정 및 실행

In [ ]:
# 다중 클래스 도로 세그멘테이션
# 차선 + 차량 + 사람 + 교통시설 + 인프라 동시 검출

import os
import json
import numpy as np
import cv2
import torch
from torch import nn
import evaluate
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import ColorJitter
from datasets import Dataset, DatasetDict
from datasets import Image as HFImage
from PIL import Image

print("🎯 다중 클래스 도로 세그멘테이션 시작!")
print("=" * 60)

# 🎨 클래스 정의 (7개 클래스: 배경 + 6개 목표 클래스)
class_map = {
    "background": 0,       # 배경
    "lane": 1,            # 차선 (흰색점선, 흰색실선, 노란색실선)
    "crosswalk": 2,       # 횡단보도
    "vehicle": 3,         # 차량 (승용차, 버스, 트럭)
    "traffic_sign": 4,    # 교통표지판
    "traffic_light": 5,   # 신호등
    "person": 6,          # 사람
    "pole": 7,            # 전봇대/기둥
}

id2label = {0: "background", 1: "lane", 2: "crosswalk", 3: "vehicle",
           4: "traffic_sign", 5: "traffic_light", 6: "person", 7: "pole"}
label2id = {v: k for k, v in id2label.items()}

# 🏷️ 라벨 매핑 정의 (정확한 매핑)
label_mapping = {
    # 차선 관련 (3개 주요 타입)
    "whdot": "lane",      # 흰색 점선 (3764개)
    "whsol": "lane",      # 흰색 실선 (410개)
    "yesol": "lane",      # 노란색 실선 (588개)
    "yedot": "lane",      # 기타 차선들도 포함
    "blsol": "lane",
    "bldot": "lane",
    "general road mark": "lane",
    "stop line": "lane",
    "guidance line": "lane",

    # 횡단보도
    "crosswalk": "crosswalk",  # 횡단보도 (1978개)

    # 차량 (3개 주요 타입)
    "car": "vehicle",          # 승용차 (1446개)
    "bus": "vehicle",          # 버스 (322개)
    "truck": "vehicle",        # 트럭 (290개)
    "bicycle": "vehicle",      # 자전거도 차량으로 포함
    "motorcycle": "vehicle",   # 오토바이도 차량으로 포함

    # 교통표지판 (별도 클래스)
    "traffic sign": "traffic_sign",  # 교통표지판 (1923개)

    # 신호등 (별도 클래스)
    "traffic light": "traffic_light",  # 신호등 (780개)

    # 사람
    "person": "person",        # 사람 (474개)
    "rider": "person",         # 탑승자도 사람으로 포함

    # 전봇대/기둥
    "pole": "pole",            # 전봇대/기둥 (3488개)
}

print("✅ 클래스 정의 완료!")
print(f"📊 총 {len(id2label)}개 클래스 (배경 + 목표 6개):")
print(f"  0: background")
print(f"  1: lane (whdot, whsol, yesol)")
print(f"  2: crosswalk")
print(f"  3: vehicle (car, bus, truck)")
print(f"  4: traffic_sign")
print(f"  5: traffic_light")
print(f"  6: person")
print(f"  7: pole")

# 경로 설정
IMAGE_DIR = '/content/sample_data/JPEGImages_mosaic'
LABEL_DIR = '/content/sample_data/labels'
MASK_DIR = '/content/sample_data/masks_multiclass'

print(f"\n📁 경로 설정:")
print(f"  - 이미지: {IMAGE_DIR}")
print(f"  - 라벨: {LABEL_DIR}")
print(f"  - 마스크: {MASK_DIR}")

# 🔧 다중 클래스 마스크 생성
print(f"\n🔧 다중 클래스 마스크 생성 중...")
os.makedirs(MASK_DIR, exist_ok=True)

# 데이터 현황 확인
if os.path.exists(IMAGE_DIR) and os.path.exists(LABEL_DIR):
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]

    image_basenames = [f.replace('_leftImg8bit.png', '') for f in all_images]
    label_basenames = [f.replace('_gtFine_polygons.json', '') for f in all_labels]
    matched_basenames = list(set(image_basenames) & set(label_basenames))

    print(f"📊 데이터 현황:")
    print(f"  - 총 이미지: {len(all_images)}개")
    print(f"  - 총 라벨: {len(all_labels)}개")
    print(f"  - 매칭된 데이터: {len(matched_basenames)}개")

    processed_count = 0
    class_stats = {class_name: 0 for class_name in class_map.keys()}

    for i, base_name in enumerate(matched_basenames):
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            continue

        height, width, _ = image.shape

        # JSON 라벨 파일 읽기
        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except:
            continue

        # 다중 클래스 마스크 생성
        mask = np.zeros((height, width), dtype=np.uint8)

        for obj in data['objects']:
            original_label = obj['label']
            points = np.array(obj['polygon'], dtype=np.int32)

            # 라벨 매핑 확인
            if original_label in label_mapping:
                mapped_class = label_mapping[original_label]
                class_id = class_map[mapped_class]

                cv2.fillPoly(mask, [points], color=class_id)
                class_stats[mapped_class] += 1

        # 마스크 저장
        mask_file_name = image_file.replace('.png', '_multiclass_mask.png')
        mask_save_path = os.path.join(MASK_DIR, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        if (i + 1) % 50 == 0:
            print(f"  진행: {i+1}/{len(matched_basenames)}")

    print(f"✅ 다중 클래스 마스크 생성 완료: {processed_count}개")

    # 클래스별 통계
    print(f"\n📊 클래스별 객체 수:")
    for class_name, count in class_stats.items():
        if count > 0:
            print(f"  - {class_name}: {count}개")

# 🗂️ 데이터셋 생성
print(f"\n🗂️ 다중 클래스 데이터셋 생성 중...")

# 파일 경로 수집
if os.path.exists(IMAGE_DIR) and os.path.exists(MASK_DIR):
    image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_multiclass_mask.png')]

    image_paths = sorted([os.path.join(IMAGE_DIR, f) for f in image_files])
    mask_paths = sorted([os.path.join(MASK_DIR, f) for f in mask_files])

    print(f"📊 파일 수집 결과:")
    print(f"  - 이미지: {len(image_paths)}개")
    print(f"  - 다중클래스 마스크: {len(mask_paths)}개")

    # 데이터셋 생성
    dataset = Dataset.from_dict({
        "image": image_paths,
        "label": mask_paths,
    })
    dataset = dataset.cast_column("image", HFImage())
    dataset = dataset.cast_column("label", HFImage())

    # 훈련/테스트 분할
    train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
    ds = DatasetDict({
        'train': train_test_split['train'],
        'test': train_test_split['test'],
    })

    print(f"✅ 데이터셋 생성 완료:")
    print(f"  - 훈련용: {len(ds['train'])}개")
    print(f"  - 테스트용: {len(ds['test'])}개")

# 🤖 모델 설정
print(f"\n🤖 다중 클래스 SegFormer 모델 설정 중...")

# 이미지 전처리기
image_processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    id2label=id2label,
    label2id=label2id,
    do_reduce_labels=False
)

# SegFormer 모델 로드 (7개 클래스용)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print(f"✅ 다중 클래스 모델 설정 완료!")
print(f"📊 모델 정보:")
print(f"  - Backbone: Mix Transformer B0")
print(f"  - 클래스 수: {len(id2label)}개")

# 🎨 색상 맵 정의 (시각화용)
color_map = {
    0: [0, 0, 0],        # background - 검은색
    1: [255, 0, 0],      # lane - 빨간색
    2: [0, 255, 0],      # crosswalk - 녹색
    3: [0, 0, 255],      # vehicle - 파란색
    4: [255, 255, 0],    # traffic_sign - 노란색
    5: [255, 165, 0],    # traffic_light - 주황색
    6: [255, 0, 255],    # person - 자홍색
    7: [0, 255, 255],    # pole - 청록색
}

print(f"\n🎨 클래스별 색상:")
for class_id, class_name in id2label.items():
    color = color_map[class_id]
    print(f"  - {class_name}: RGB{color}")

print(f"\n✨ 다중 클래스 세그멘테이션 준비 완료!")
print(f"🎯 이제 다음 단계들을 실행하세요:")
print(f"  1. 훈련 파라미터 설정")
print(f"  2. 실제 모델 훈련")
print(f"  3. 다중 클래스 결과 시각화")

# 다음 단계를 위한 변수들
print(f"\n💾 준비된 변수들:")
print(f"ds = 다중클래스 데이터셋")
print(f"model = 7개 클래스 SegFormer 모델")
print(f"id2label = 클래스 매핑")
print(f"color_map = 시각화용 색상 맵")

2단계

In [ ]:
# 2단계: 데이터셋 생성 및 분할 (경로 수정된 버전)
# 이미지와 마스크를 Hugging Face 데이터셋으로 변환

# 딥러닝 라이브러리 설치
!pip install -q transformers datasets evaluate accelerate

# 필요한 라이브러리 임포트
import os
from datasets import Dataset, DatasetDict
from datasets import Image as HFImage
from PIL import Image

print("딥러닝 라이브러리 로드 완료!")

# 🔍 실제 경로 자동 탐지
print("\n🔍 실제 폴더 위치 확인 중...")

# 가능한 이미지 폴더 경로들
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]

IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        png_files = [f for f in os.listdir(path) if f.endswith('.png')]
        print(f"✅ 이미지 폴더 찾음: {path}")
        print(f"   📁 PNG 파일 수: {len(png_files)}개")
        break

if IMAGE_DIR is None:
    print("❌ 기본 경로에서 이미지 폴더를 찾을 수 없습니다!")
    print("🔍 sample_data 하위 폴더 전체 탐색:")

    for item in os.listdir('/content/sample_data/'):
        item_path = os.path.join('/content/sample_data/', item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                files = os.listdir(item_path)
                png_count = len([f for f in files if f.endswith('.png')])
                if png_count > 0:
                    print(f"     🖼️ PNG 파일 {png_count}개 발견!")
                    IMAGE_DIR = item_path
                    break
            except:
                pass

# 마스크 폴더는 고정
MASK_DIR = '/content/sample_data/masks_final'

print(f"\n📁 최종 경로 설정:")
print(f"  - 이미지: {IMAGE_DIR}")
print(f"  - 마스크: {MASK_DIR}")

# 경로 유효성 확인
if IMAGE_DIR is None or not os.path.exists(IMAGE_DIR):
    print("\n❌ 이미지 폴더를 찾을 수 없습니다!")
    print("💡 해결 방법:")
    print("1. 1단계(압축 해제 및 마스크 생성)를 먼저 실행하세요")
    print("2. 또는 다음 명령어로 수동 압축 해제:")
    print("   !cd /content/sample_data && unzip -q MonoCameraSemanticSegmentation.zip")
    exit()

if not os.path.exists(MASK_DIR):
    print(f"\n❌ 마스크 폴더가 없습니다: {MASK_DIR}")
    print("💡 1단계(마스크 생성)를 먼저 실행하세요!")
    exit()

# 클래스 정의
id2label = {0: "background", 1: "lane"}
label2id = {v: k for k, v in id2label.items()}

print("클래스 설정:")
print(f"  - {id2label}")

# 이미지와 마스크 파일 경로 수집 (전체 데이터 사용)
print("\n파일 경로 수집 중...")

# 모든 이미지와 마스크 파일 수집 (Daeduk + SangamDMC)
if os.path.exists(IMAGE_DIR):
    image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
else:
    image_files = []

if os.path.exists(MASK_DIR):
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_mask.png')]
else:
    mask_files = []

# 전체 경로로 변환
image_paths = sorted([os.path.join(IMAGE_DIR, f) for f in image_files])
mask_paths = sorted([os.path.join(MASK_DIR, f) for f in mask_files])

# 데이터 현황 분석
daeduk_images = [p for p in image_paths if 'Daeduk' in os.path.basename(p)]
sangam_images = [p for p in image_paths if 'SangamDMC' in os.path.basename(p)]
daeduk_masks = [p for p in mask_paths if 'Daeduk' in os.path.basename(p)]
sangam_masks = [p for p in mask_paths if 'SangamDMC' in os.path.basename(p)]

print(f"📊 수집된 파일 (전체 데이터):")
print(f"  🖼️ 이미지: {len(image_paths)}개")
print(f"    - 🟦 Daeduk: {len(daeduk_images)}개")
print(f"    - 🟩 SangamDMC: {len(sangam_images)}개")
print(f"  🎭 마스크: {len(mask_paths)}개")
print(f"    - 🟦 Daeduk: {len(daeduk_masks)}개")
print(f"    - 🟩 SangamDMC: {len(sangam_masks)}개")

# 파일 수가 일치하는지 확인
if len(image_paths) != len(mask_paths):
    print("⚠️  경고: 이미지와 마스크 파일 수가 일치하지 않습니다!")
    print(f"이미지: {len(image_paths)}개, 마스크: {len(mask_paths)}개")

# 데이터가 없으면 중단
if len(image_paths) == 0 or len(mask_paths) == 0:
    print("\n❌ 데이터가 없습니다!")
    print(f"이미지: {len(image_paths)}개, 마스크: {len(mask_paths)}개")
    print("💡 1단계를 먼저 실행해서 데이터를 준비하세요!")
    exit()

# 데이터셋 생성
print("\n데이터셋 생성 중...")
dataset = Dataset.from_dict({
    "image": image_paths,
    "label": mask_paths,
})

# 이미지 타입으로 변환 (Hugging Face 형식)
dataset = dataset.cast_column("image", HFImage())
dataset = dataset.cast_column("label", HFImage())

print("✅ 기본 데이터셋 생성 완료!")

# 훈련/테스트 분할 (8:2 비율)
print("\n데이터셋 분할 중...")
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)

# 최종 데이터셋 구성
ds = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test'],
})

print("\n🎉 데이터셋 분할 완료!")
print(f"📊 최종 데이터셋 구성:")
print(f"  - 훈련용: {len(ds['train'])}개")
print(f"  - 테스트용: {len(ds['test'])}개")
print(f"  - 총합: {len(dataset)}개")

# 데이터셋 구조 확인
print(f"\n📋 데이터셋 구조:")
print(ds)

# 샘플 데이터 확인
if len(ds['train']) > 0:
    print(f"\n🔍 샘플 데이터 확인:")
    sample = ds['train'][0]
    print(f"  - 이미지 크기: {sample['image'].size}")
    print(f"  - 라벨 크기: {sample['label'].size}")
else:
    print(f"\n❌ 훈련 데이터가 없습니다!")

print("\n✨ 2단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들 저장
print("\n💾 다음 단계를 위한 변수들:")
print("ds = 생성된 데이터셋")
print("id2label, label2id = 클래스 매핑")
print(f"실제 데이터 수: 훈련 {len(ds['train'])}개, 테스트 {len(ds['test'])}개")

3단

In [ ]:
# 3단계: 모델 및 전처리 설정
# SegFormer 모델과 이미지 전처리 파이프라인 구성

# 필요한 라이브러리 임포트
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from torchvision.transforms import ColorJitter
import torch

print("모델 라이브러리 로드 완료!")

# 클래스 설정 (2단계에서 가져옴)
id2label = {0: "background", 1: "lane"}
label2id = {v: k for k, v in id2label.items()}

print("클래스 매핑:")
print(f"  - ID to Label: {id2label}")
print(f"  - Label to ID: {label2id}")

# SegFormer 이미지 전처리기 설정
print("\n이미지 전처리기 설정 중...")
try:
    image_processor = SegformerImageProcessor.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512",
        id2label=id2label,
        label2id=label2id,
        do_reduce_labels=False  # 라벨 자동 변환 비활성화
    )
    print("✅ 이미지 전처리기 설정 완료!")
except Exception as e:
    print(f"❌ 전처리기 설정 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

# 데이터 증강 설정 (훈련 시에만 사용)
print("\n데이터 증강 설정 중...")
jitter = ColorJitter(
    brightness=0.25,  # 밝기 변화
    contrast=0.25,    # 대비 변화
    saturation=0.25,  # 채도 변화
    hue=0.1          # 색조 변화
)

def train_transforms(example_batch):
    """훈련용 데이터 변환 (데이터 증강 포함)"""
    # 이미지에 데이터 증강 적용
    images = [jitter(x.convert("RGB")) for x in example_batch['image']]
    # 라벨은 그대로 유지
    labels = [x for x in example_batch['label']]

    # 전처리기로 배치 처리
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

def val_transforms(example_batch):
    """검증용 데이터 변환 (데이터 증강 없음)"""
    # 이미지를 RGB로만 변환
    images = [x.convert("RGB") for x in example_batch['image']]
    # 라벨은 그대로 유지
    labels = [x for x in example_batch['label']]

    # 전처리기로 배치 처리
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

print("✅ 데이터 변환 함수 정의 완료!")

# SegFormer 모델 로드
print("\n🤖 SegFormer 모델 로드 중...")
try:
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b0",  # Mix Transformer B0 backbone 사용
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # 클래스 수가 다를 때 크기 무시
    )
    print("✅ SegFormer 모델 로드 완료!")

    # 모델 정보 출력
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"📊 모델 정보:")
    print(f"  - Backbone: Mix Transformer B0")
    print(f"  - 클래스 수: {len(id2label)}개")
    print(f"  - 총 파라미터: {total_params/1e6:.1f}M개")
    print(f"  - 훈련 가능 파라미터: {trainable_params/1e6:.1f}M개")

except Exception as e:
    print(f"❌ 모델 로드 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🔧 사용 장치: {device}")

if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name()}")
    print(f"  - CUDA 버전: {torch.version.cuda}")

    # GPU 메모리 정보
    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = gpu_memory / 1e9
    print(f"  - GPU 메모리: {gpu_memory_gb:.1f} GB")

    # 권장 배치 크기
    if gpu_memory_gb >= 16:
        recommended_batch = 4
    elif gpu_memory_gb >= 8:
        recommended_batch = 2
    else:
        recommended_batch = 1
    print(f"  - 권장 배치 크기: {recommended_batch}")

else:
    print("  - CPU 모드로 실행됩니다 (훈련이 매우 느릴 수 있습니다)")

# 모델을 GPU로 이동 (가능한 경우)
if torch.cuda.is_available():
    try:
        model = model.to(device)
        print("✅ 모델을 GPU로 이동 완료!")
    except Exception as e:
        print(f"⚠️ GPU 이동 실패, CPU 사용: {e}")

print("\n✨ 3단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들
print("\n💾 다음 단계를 위한 주요 변수들:")
print("model = SegFormer 모델")
print("image_processor = 이미지 전처리기")
print("train_transforms, val_transforms = 데이터 변환 함수")
print("id2label, label2id = 클래스 매핑")
print(f"device = {device}")

# 메모리 최적화 팁
print(f"\n💡 메모리 최적화 팁:")
print(f"  - GPU 메모리 부족 시: 배치 크기를 1로 줄이세요")
print(f"  - 훈련 속도 향상: eval_steps를 늘리세요 (예: 500)")
print(f"  - 더 정확한 평가: eval_steps를 줄이세요 (예: 100)")

# 다음 단계 안내
print(f"\n🎯 다음 단계 안내:")
print(f"  4단계: 훈련 설정 및 메트릭")
print(f"  5단계: 실제 모델 훈련")

In [ ]:
# 4단계: 훈련 설정 및 메트릭
# 평가 메트릭과 훈련 파라미터 설정

# 필요한 라이브러리 설치 및 임포트
!pip install -q evaluate torch torchvision

import torch
from torch import nn
import evaluate
from transformers import TrainingArguments, Trainer

print("훈련 라이브러리 로드 완료!")

# 평가 메트릭 설정
print("\n📊 평가 메트릭 설정 중...")
try:
    metric = evaluate.load("mean_iou")
    print("✅ mIoU 메트릭 로드 완료!")
except Exception as e:
    print(f"❌ 메트릭 로드 오류: {e}")
    print("인터넷 연결을 확인하고 다시 시도하세요.")

def compute_metrics(eval_pred):
    """모델 성능 평가 함수"""
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)

        # 예측 결과를 원본 이미지 크기로 리사이즈
        logits_resized = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],  # (height, width)
            mode="bilinear",
            align_corners=False,
        )

        # 가장 높은 확률의 클래스를 예측값으로 선택
        pred_labels = logits_resized.argmax(dim=1).numpy()

        # mIoU 및 정확도 계산
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label),
            ignore_index=255,  # 무시할 픽셀 인덱스
            reduce_labels=False,
        )

        return {
            "mean_iou": metrics["mean_iou"],
            "mean_accuracy": metrics["mean_accuracy"],
        }

print("✅ 평가 메트릭 설정 완료!")

# 훈련 파라미터 설정
print("\n⚙️ 훈련 파라미터 설정 중...")

# GPU 메모리에 따른 배치 크기 자동 조정
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory >= 16:
        batch_size = 4
        print(f"🔧 고용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
    elif gpu_memory >= 8:
        batch_size = 2
        print(f"🔧 중용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
    else:
        batch_size = 1
        print(f"🔧 저용량 GPU 감지 ({gpu_memory:.1f}GB): 배치 크기 {batch_size}")
else:
    batch_size = 1
    print("🔧 CPU 모드: 배치 크기 1")

training_args = TrainingArguments(
    # 기본 설정
    output_dir="./segformer-lane-detection",

    # 학습률 및 에포크
    learning_rate=6e-5,
    num_train_epochs=20,

    # 배치 크기 (GPU 메모리에 따라 자동 조정)
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    # 저장 설정
    save_total_limit=3,  # 최대 3개 체크포인트만 유지
    save_strategy="steps",
    save_steps=200,

    # 평가 설정 (수정된 파라미터명)
    eval_strategy="steps",  # evaluation_strategy → eval_strategy
    eval_steps=200,

    # 로깅
    logging_dir="./logs",
    logging_steps=100,

    # 기타 설정
    remove_unused_columns=False,  # 사용하지 않는 컬럼 제거 안함
    push_to_hub=False,  # Hugging Face Hub에 업로드 안함

    # 성능 최적화
    dataloader_pin_memory=True,
    dataloader_num_workers=2,

    # 조기 종료 (옵션)
    load_best_model_at_end=True,
    metric_for_best_model="mean_iou",
    greater_is_better=True,

    # 추가 설정
    report_to=None,  # wandb 등 로깅 비활성화
    seed=42,  # 재현 가능한 결과를 위한 시드
)

print("✅ 훈련 파라미터 설정 완료!")

# 설정 요약 출력
print(f"\n📋 주요 훈련 설정:")
print(f"  - 학습률: {training_args.learning_rate}")
print(f"  - 에포크: {training_args.num_train_epochs}")
print(f"  - 배치 크기: {training_args.per_device_train_batch_size}")
print(f"  - 평가 주기: {training_args.eval_steps} 스텝마다")
print(f"  - 저장 위치: {training_args.output_dir}")
print(f"  - 로그 위치: {training_args.logging_dir}")

# 예상 훈련 시간 계산 (대략적)
if 'ds' in globals():
    total_steps = (len(ds['train']) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
    estimated_time = total_steps * 2 / 60  # 대략 스텝당 2초 가정
    print(f"  - 예상 총 스텝: {total_steps}")
    print(f"  - 예상 훈련 시간: 약 {estimated_time:.0f}분")
else:
    print(f"  - ds 변수가 없어서 시간 계산 불가")

# 메모리 최적화 및 성능 팁
print(f"\n💡 메모리 최적화 팁:")
print(f"  - GPU 메모리 부족 시: 배치 크기를 1로 줄이세요")
print(f"  - 더 빠른 훈련: eval_steps를 500으로 늘리세요")
print(f"  - 더 정확한 평가: eval_steps를 100으로 줄이세요")
print(f"  - 디스크 공간 절약: save_total_limit를 1로 줄이세요")

# GPU 메모리 정보 (사용 가능한 경우)
if torch.cuda.is_available():
    print(f"\n🔧 현재 GPU 상태:")
    print(f"  - 총 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if hasattr(torch.cuda, 'mem_get_info'):
        free_memory, total_memory = torch.cuda.mem_get_info()
        print(f"  - 사용 가능: {free_memory / 1e9:.1f} GB")
        print(f"  - 사용 중: {(total_memory - free_memory) / 1e9:.1f} GB")

print("\n✨ 4단계 완료! 다음 단계로 진행하세요.")

# 다음 단계에서 사용할 변수들
print("\n💾 다음 단계를 위한 주요 변수들:")
print("training_args = 훈련 파라미터")
print("compute_metrics = 평가 함수")
print("metric = mIoU 메트릭")

# 다음 단계 안내
print(f"\n🎯 다음 단계 안내:")
print(f"  5단계: 실제 모델 훈련 및 평가")
print(f"  - 데이터셋 전처리 적용")
print(f"  - Trainer 설정")
print(f"  - 훈련 실행")
print(f"  - 최종 평가 및 모델 저장")

# 훈련 전 체크리스트
print(f"\n✅ 훈련 전 체크리스트:")
required_vars = ['ds', 'model', 'image_processor', 'train_transforms', 'val_transforms', 'id2label']
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    print(f"❌ 누락된 변수들: {missing_vars}")
    print(f"이전 단계들을 먼저 실행해주세요!")
else:
    print(f"✅ 모든 필수 변수가 준비되었습니다!")
    print(f"5단계를 실행할 수 있습니다!")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# wandb 충돌 해결
import os
if "WANDB_DISABLED" in os.environ:
    del os.environ["WANDB_DISABLED"]

# 수정된 training_args로 다시 설정
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./segformer-lane-detection",
    learning_rate=6e-5,
    num_train_epochs=30,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",  # 🔧 이 줄이 핵심!
    seed=42,
)

print("✅ training_args 수정 완료!")

! 🚨 문제를 발견했어요!
클래스 수: 2개 (배경, 차선) ← 이게 문제에요!
🔍 원인:

다중클래스 모델(8개)을 만들었지만
기존 2클래스 변수(id2label)가 남아있음
5단계 코드가 기존 변수를 참조하고 있음

In [ ]:
# 현재 id2label 확인
print("현재 id2label:", id2label)
print("현재 클래스 수:", len(id2label))

# 8개 클래스로 다시 정의
id2label = {0: "background", 1: "lane", 2: "crosswalk", 3: "vehicle",
           4: "traffic_sign", 5: "traffic_light", 6: "person", 7: "pole"}
label2id = {v: k for k, v in id2label.items()}

print("수정된 id2label:", id2label)
print("수정된 클래스 수:", len(id2label))

가중치 변경
class_weights = {
    "background": 0.1,     # 거의 무시
    "lane": 1.0,          # 기준값  
    "crosswalk": 2.0,     # 2배
    "vehicle": 3.0,       # 3배 ⭐
    "traffic_sign": 2.5,  # 2.5배
    "traffic_light": 5.0, # 5배 ⭐⭐
    "person": 8.0,        # 8배 ⭐⭐⭐
    "pole": 1.5,          # 1.5배
}

In [ ]:
#4단계
# 클래스 균형 맞춘 훈련 설정
# 차선 외에 차량, 사람도 잘 인식하도록 개선

import torch
from torch import nn
import evaluate
from transformers import TrainingArguments, Trainer
import numpy as np

print("🔧 클래스 균형 맞춘 훈련 설정 시작!")

# 클래스별 데이터 수 (분석 결과 기반)
class_counts = {
    0: 1000000,  # background (추정)
    1: 4762,     # lane (whdot + whsol + yesol)
    2: 1978,     # crosswalk
    3: 2058,     # vehicle (car + bus + truck)
    4: 1923,     # traffic_sign
    5: 780,      # traffic_light
    6: 474,      # person (가장 적음!)
    7: 3488,     # pole
}

# 🎯 수동 클래스 가중치 설정 (더 강력하게)
print("🎯 수동 클래스 가중치 설정 중...")

# 수동으로 가중치 조정 (중요한 클래스 강화)
class_weights_manual = {
    0: 0.1,    # background - 매우 낮음
    1: 1.0,    # lane - 기준값
    2: 2.0,    # crosswalk - 2배
    3: 3.0,    # vehicle - 3배 강화 ⭐
    4: 2.5,    # traffic_sign - 2.5배
    5: 5.0,    # traffic_light - 5배 강화 ⭐⭐
    6: 8.0,    # person - 8배 강화 ⭐⭐⭐
    7: 1.5,    # pole - 1.5배
}

class_weights = [class_weights_manual[i] for i in range(len(id2label))]

print("📊 수동 조정된 클래스별 가중치:")
for i, (class_id, class_name) in enumerate(id2label.items()):
    weight = class_weights[i]
    stars = "⭐" * min(int(weight), 5)  # 별표로 중요도 표시
    print(f"  {class_name}: {weight:.1f} {stars}")

print(f"\n💡 가중치 의미:")
print(f"  - 높을수록 해당 클래스 놓쳤을 때 더 큰 벌점")
print(f"  - person(8.0): 사람 놓치면 8배 큰 손실")
print(f"  - traffic_light(5.0): 신호등 놓치면 5배 큰 손실")
print(f"  - vehicle(3.0): 차량 놓치면 3배 큰 손실")

# PyTorch 텐서로 변환
class_weights_tensor = torch.FloatTensor(class_weights)
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

# 🔧 커스텀 Trainer 클래스 정의
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')

        # 손실 함수에 클래스 가중치 적용
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=255)

        # logits를 올바른 크기로 리사이즈
        upsampled_logits = nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        )

        # 손실 계산
        loss = loss_fct(upsampled_logits, labels.long())

        return (loss, outputs) if return_outputs else loss

print("✅ 가중치 적용 Trainer 클래스 정의 완료!")

# 📊 개선된 평가 메트릭
metric = evaluate.load("mean_iou")

def compute_metrics_detailed(eval_pred):
    """클래스별 상세 성능 평가"""
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)

        # 리사이즈
        logits_resized = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        pred_labels = logits_resized.argmax(dim=1).numpy()

        # 전체 성능
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label),
            ignore_index=255,
            reduce_labels=False,
        )

        # 클래스별 IoU 계산
        class_ious = {}
        for class_id, class_name in id2label.items():
            if class_id == 0:  # background 제외
                continue

            # 해당 클래스만 추출
            pred_class = (pred_labels == class_id)
            true_class = (labels == class_id)

            # IoU 계산
            intersection = np.logical_and(pred_class, true_class).sum()
            union = np.logical_or(pred_class, true_class).sum()

            if union > 0:
                iou = intersection / union
                class_ious[f"iou_{class_name}"] = iou
            else:
                class_ious[f"iou_{class_name}"] = 0.0

        # 전체 결과 합치기
        result = {
            "mean_iou": metrics["mean_iou"],
            "mean_accuracy": metrics["mean_accuracy"],
        }
        result.update(class_ious)

        return result

print("✅ 상세 평가 메트릭 설정 완료!")

# ⚙️ 개선된 훈련 파라미터
training_args_balanced = TrainingArguments(
    output_dir="./segformer-multiclass-balanced",

    # 학습률 (좀 더 낮게)
    learning_rate=3e-5,  # 6e-5 → 3e-5로 감소

    # 에포크 (충분히)
    num_train_epochs=30,

    # 배치 크기
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # 저장 및 평가 (더 자주)
    save_total_limit=5,
    save_strategy="steps",
    save_steps=100,  # 200 → 100 (더 자주 저장)

    eval_strategy="steps",
    eval_steps=100,   # 200 → 100 (더 자주 평가)

    # 로깅
    logging_steps=50,  # 100 → 50 (더 자주 로깅)

    # 기타
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",
    seed=42,

    # 조기 종료 (성능 개선)
    load_best_model_at_end=True,
    metric_for_best_model="mean_iou",
    greater_is_better=True,

    # 학습률 스케줄러
    warmup_steps=100,
    lr_scheduler_type="cosine",
)

print("✅ 개선된 훈련 파라미터 설정 완료!")

# 🎯 WeightedTrainer 생성
print("\n🎯 균형 잡힌 Trainer 설정 중...")

trainer = WeightedTrainer(
    model=model,
    args=training_args_balanced,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    compute_metrics=compute_metrics_detailed,
)

print("✅ WeightedTrainer 설정 완료!")

# 📊 훈련 정보
print(f"\n📊 개선된 훈련 정보:")
print(f"  - 모델: SegFormer (8개 클래스)")
print(f"  - 클래스 가중치 적용: ✅")
print(f"  - 학습률: {training_args_balanced.learning_rate}")
print(f"  - 에포크: {training_args_balanced.num_train_epochs}")
print(f"  - 평가 주기: {training_args_balanced.eval_steps} 스텝마다")

print(f"\n💡 개선 사항:")
print(f"  ✅ 클래스 가중치로 불균형 해결")
print(f"  ✅ 낮은 학습률로 안정적 학습")
print(f"  ✅ 더 자주 평가/저장")
print(f"  ✅ 클래스별 상세 성능 측정")

print(f"\n🚀 이제 다음 명령어로 훈련을 시작하세요:")
print(f"trainer.train()")

# 🎨 시각화용 개선된 색상 맵
improved_color_map = {
    0: [0, 0, 0],          # background - 투명
    1: [255, 0, 0],        # lane - 빨간색 (밝게)
    2: [0, 255, 0],        # crosswalk - 녹색 (밝게)
    3: [0, 100, 255],      # vehicle - 파란색 (더 진하게)
    4: [255, 255, 0],      # traffic_sign - 노란색 (밝게)
    5: [255, 165, 0],      # traffic_light - 주황색 (밝게)
    6: [255, 20, 147],     # person - 핫핑크 (더 눈에 띄게)
    7: [0, 255, 255],      # pole - 청록색
}

print(f"\n🎨 개선된 시각화 색상:")
for class_id, class_name in id2label.items():
    color = improved_color_map[class_id]
    if class_id == 6:  # person
        print(f"  - {class_name}: RGB{color} ⭐ (강조색)")
    elif class_id == 3:  # vehicle
        print(f"  - {class_name}: RGB{color} ⭐ (강조색)")
    else:
        print(f"  - {class_name}: RGB{color}")

print(f"\n✨ 클래스 균형 설정 완료!")
print(f"이제 차량과 사람도 잘 인식할 거예요! 🚗👥")

🎯 성능 평가:
**mIoU 69%**는 차선 검출에서:

🟡 괜찮은 성능 (보통 60-80% 범위)
🟢 실용적으로 사용 가능한 수준
🔴 더 개선 여지 있음

💡 성능 개선 방법:

더 많은 에포크: 10 → 20으로 늘리기
학습률 조정: 6e-5 → 3e-5로 줄이기
데이터 증강 강화: 더 다양한 변형
더 큰 모델: mit-b0 → mit-b1 사용

📊 핵심 개선사항:
1. 차선 가중치 강화 ⭐⭐⭐

lane: 3.0 (작은 whdot 점선 보강)

2. 좌우 반전 증강 🔄

50% 확률로 좌우 반전
왼쪽 편향 문제 해결

3. 멀티스케일 손실 🔍

작은 객체(whdot) 검출 강화
30% 확률로 다운샘플 손실 추가

4. 회전 변환 🌀

30% 확률로 ±5도 회전
다양한 각도의 차선 학습

In [ ]:
# 4단계
#분석 결과 기반 개선된 훈련 설정
# 작은 차선(whdot)과 위치별 불균형 해결

import torch
from torch import nn
import evaluate
from transformers import TrainingArguments, Trainer
from torchvision.transforms import ColorJitter
import random
from PIL import Image
import numpy as np

print("🔧 분석 결과 기반 개선된 훈련 설정!")

# 🎯 차선 타입별 특화 가중치
print("🎯 차선 특성 기반 가중치 설정...")

class_weights_improved = {
    0: 0.1,    # background
    1: 3.0,    # lane - 작은 whdot 때문에 높게! ⭐⭐⭐
    2: 12.0,    # crosswalk
    3: 5.0,    # vehicle
    4: 7.5,    # traffic_sign
    5: 10.0,    # traffic_light
    6: 18.0,    # person
    7: 3.5,    # pole
}

class_weights = [class_weights_improved[i] for i in range(len(id2label))]

print("📊 차선 특성 고려한 가중치:")
for i, (class_id, class_name) in enumerate(id2label.items()):
    weight = class_weights[i]
    if class_name == "lane":
        print(f"  {class_name}: {weight:.1f} ⭐⭐⭐ (작은 whdot 보강)")
    else:
        stars = "⭐" * min(int(weight), 5)
        print(f"  {class_name}: {weight:.1f} {stars}")

# PyTorch 텐서로 변환
class_weights_tensor = torch.FloatTensor(class_weights)
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

# 🔄 개선된 데이터 증강 (좌우 반전 + 멀티스케일)
jitter = ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)

def train_transforms_improved(example_batch):
    """개선된 데이터 증강: 좌우 반전 + 강화된 증강"""
    images = []
    labels = []

    for img, label in zip(example_batch['image'], example_batch['label']):
        # 1. 50% 확률로 좌우 반전 (위치 불균형 해결)
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            label = label.transpose(Image.FLIP_LEFT_RIGHT)

        # 2. 30% 확률로 약간 회전 (다양한 각도의 차선)
        if random.random() < 0.3:
            angle = random.uniform(-5, 5)  # ±5도
            img = img.rotate(angle, fillcolor=(0, 0, 0))
            label = label.rotate(angle, fillcolor=0)

        # 3. 색상 증강 (더 강하게)
        img = jitter(img.convert("RGB"))

        images.append(img)
        labels.append(label)

    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

def val_transforms_improved(example_batch):
    """검증용: 증강 없음"""
    images = [x.convert("RGB") for x in example_batch['image']]
    labels = [x for x in example_batch['label']]
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

print("✅ 개선된 데이터 증강 설정 완료!")
print("  - ✅ 좌우 반전 50%")
print("  - ✅ 회전 변환 30%")
print("  - ✅ 강화된 색상 증강")

# 🏋️ 멀티스케일 손실 함수 (작은 객체 강화)
class ImprovedWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')

        # 멀티스케일 손실 계산
        total_loss = 0

        # 1. 원본 크기 손실
        upsampled_logits = nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        )

        # 클래스 가중치 적용
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=255)
        main_loss = loss_fct(upsampled_logits, labels.long())
        total_loss += main_loss

        # 2. 다운샘플 손실 (작은 객체 강화)
        if random.random() < 0.3:  # 30% 확률로 적용
            # 절반 크기로 다운샘플
            small_labels = nn.functional.interpolate(
                labels.float().unsqueeze(1),
                scale_factor=0.5,
                mode="nearest"
            ).squeeze(1).long()

            small_logits = nn.functional.interpolate(
                logits,
                size=small_labels.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            small_loss = loss_fct(small_logits, small_labels)
            total_loss += 0.3 * small_loss  # 30% 가중치

        return (total_loss, outputs) if return_outputs else total_loss

print("✅ 멀티스케일 손실 함수 설정 완료!")

# 📊 더 상세한 평가 메트릭
metric = evaluate.load("mean_iou")

def compute_metrics_advanced(eval_pred):
    """차선 타입별 상세 분석"""
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)

        logits_resized = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        pred_labels = logits_resized.argmax(dim=1).numpy()

        # 전체 성능
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label),
            ignore_index=255,
            reduce_labels=False,
        )

        # 클래스별 상세 분석
        class_metrics = {}
        for class_id, class_name in id2label.items():
            if class_id == 0:  # background 제외
                continue

            pred_class = (pred_labels == class_id)
            true_class = (labels == class_id)

            # IoU 계산
            intersection = np.logical_and(pred_class, true_class).sum()
            union = np.logical_or(pred_class, true_class).sum()

            if union > 0:
                iou = intersection / union
                class_metrics[f"iou_{class_name}"] = iou

                # 정밀도와 재현율도 계산
                if pred_class.sum() > 0:
                    precision = intersection / pred_class.sum()
                    class_metrics[f"precision_{class_name}"] = precision

                if true_class.sum() > 0:
                    recall = intersection / true_class.sum()
                    class_metrics[f"recall_{class_name}"] = recall
            else:
                class_metrics[f"iou_{class_name}"] = 0.0

        result = {
            "mean_iou": metrics["mean_iou"],
            "mean_accuracy": metrics["mean_accuracy"],
        }
        result.update(class_metrics)

        return result

print("✅ 고급 평가 메트릭 설정 완료!")

# ⚙️ 최적화된 훈련 파라미터
training_args_optimized = TrainingArguments(
    output_dir="./segformer-multiclass-optimized",

    # 학습률 (더 정교하게)
    learning_rate=2e-5,  # 더 낮게

    # 에포크
    num_train_epochs=40,  # 더 오래

    # 배치 크기
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # 더 자주 평가/저장
    save_total_limit=5,
    save_strategy="steps",
    save_steps=50,   # 더 자주

    eval_strategy="steps",
    eval_steps=50,   # 더 자주

    # 로깅
    logging_steps=25,

    # 기타
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",
    seed=42,

    # 학습률 스케줄러
    warmup_steps=200,
    lr_scheduler_type="cosine",

    # 조기 종료
    load_best_model_at_end=True,
    metric_for_best_model="iou_lane",  # 차선 IoU 기준!
    greater_is_better=True,
)

print("✅ 최적화된 훈련 파라미터 설정 완료!")

# 🎯 최종 Trainer 설정
print("\n🎯 최종 개선된 Trainer 설정 중...")

# 데이터셋에 개선된 변환 적용
ds["train"].set_transform(train_transforms_improved)
ds["test"].set_transform(val_transforms_improved)

trainer = ImprovedWeightedTrainer(
    model=model,
    args=training_args_optimized,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    compute_metrics=compute_metrics_advanced,
)

print("✅ 최종 Trainer 설정 완료!")

print(f"\n📊 최종 개선 사항:")
print(f"  ✅ 차선 가중치 3.0 (작은 whdot 보강)")
print(f"  ✅ 좌우 반전 50% (위치 불균형 해결)")
print(f"  ✅ 회전 변환 30% (다양한 각도)")
print(f"  ✅ 멀티스케일 손실 (작은 객체 강화)")
print(f"  ✅ 더 낮은 학습률 (2e-5)")
print(f"  ✅ 더 많은 에포크 (40)")
print(f"  ✅ 차선 IoU 기준 조기종료")

print(f"\n🚀 이제 다음 명령어로 훈련을 시작하세요:")
print(f"trainer.train()")

print(f"\n🎯 예상 개선 효과:")
print(f"  - 왼쪽 차선 인식 개선")
print(f"  - 작은 whdot 점선 인식 개선")
print(f"  - 전체적인 차선 검출 정확도 향상")

print(f"\n✨ 분석 기반 최적화 설정 완료!")

In [ ]:
# 5단계: 훈련 실행 (간단 버전)
# 모든 설정을 통합하여 실제 훈련 시작

print("🔍 사전 확인 중...")

# 필수 변수들이 모두 있는지 확인
required_vars = ['ds', 'model', 'training_args', 'compute_metrics', 'train_transforms', 'val_transforms']
missing_vars = []

for var_name in required_vars:
    if var_name not in globals():
        missing_vars.append(var_name)

if missing_vars:
    print(f"❌ 누락된 변수들: {missing_vars}")
    print("이전 단계들을 먼저 실행해주세요!")
    print("필요한 단계:")
    if 'ds' in missing_vars:
        print("  - 2단계: 데이터셋 생성")
    if 'model' in missing_vars:
        print("  - 3단계: 모델 설정")
    if 'training_args' in missing_vars:
        print("  - 4단계: 훈련 설정")
else:
    print("✅ 모든 필수 변수들이 준비되었습니다!")

# 데이터셋에 전처리 함수 적용
print("\n📦 데이터셋 전처리 적용 중...")
ds["train"].set_transform(train_transforms)
ds["test"].set_transform(val_transforms)
print("✅ 데이터셋 전처리 적용 완료!")

# Trainer 설정
print("\n🎯 Trainer 설정 중...")
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    compute_metrics=compute_metrics,
)

print("✅ Trainer 설정 완료!")

# 훈련 시작 전 최종 정보 출력
print(f"\n📊 훈련 시작 전 최종 정보:")
print(f"  - 모델: SegFormer (MiT-B0)")
print(f"  - 훈련 데이터: {len(ds['train'])}개")
print(f"  - 테스트 데이터: {len(ds['test'])}개")
print(f"  - 클래스 수: {len(id2label)}개 (배경, 차선)")
print(f"  - 에포크: {training_args.num_train_epochs}")
print(f"  - 배치 크기: {training_args.per_device_train_batch_size}")

# 예상 훈련 시간 계산
total_steps = (len(ds['train']) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
estimated_time = total_steps * 2 / 60  # 대략 스텝당 2초 가정
print(f"  - 예상 총 스텝: {total_steps}")
print(f"  - 예상 훈련 시간: 약 {estimated_time:.0f}분")

# 🚀 훈련 시작
print(f"\n🚀 훈련 시작!")
print("=" * 50)

try:
    # 실제 훈련 실행
    trainer.train()

    print("\n" + "=" * 50)
    print("🎉 훈련 완료!")

    # 최종 평가
    print(f"\n📊 최종 평가 중...")
    eval_results = trainer.evaluate()

    print(f"✅ 최종 평가 결과:")
    for key, value in eval_results.items():
        if 'eval_' in key:
            metric_name = key.replace('eval_', '')
            print(f"  - {metric_name}: {value:.4f}")

    # 모델 저장
    print(f"\n💾 모델 저장 중...")
    trainer.save_model()
    print(f"✅ 모델 저장 완료: {training_args.output_dir}")

    # 성능 해석
    final_iou = eval_results.get('eval_mean_iou', 0)
    final_acc = eval_results.get('eval_mean_accuracy', 0)

    print(f"\n🎯 성능 해석:")
    if final_iou > 0.7:
        print(f"  🟢 훌륭한 성능! (mIoU: {final_iou:.3f})")
    elif final_iou > 0.5:
        print(f"  🟡 괜찮은 성능! (mIoU: {final_iou:.3f})")
    else:
        print(f"  🔴 추가 훈련 필요 (mIoU: {final_iou:.3f})")

    print(f"\n📁 결과물:")
    print(f"  - 훈련된 모델: {training_args.output_dir}")
    print(f"  - 로그 파일: ./logs")

    # 성능 개선 팁
    print(f"\n💡 성능 개선 팁:")
    if final_iou < 0.6:
        print(f"  - 더 많은 에포크로 재훈련")
        print(f"  - 학습률 조정 (현재: {training_args.learning_rate})")
        print(f"  - 데이터 증강 강화")

except Exception as e:
    print(f"\n❌ 훈련 중 오류 발생:")
    print(f"Error: {str(e)}")
    print(f"\n💡 해결 방법:")
    print(f"  1. GPU 메모리 부족 → 배치 크기를 1로 줄이기")
    print(f"  2. CUDA 오류 → 런타임 재시작")
    print(f"  3. 데이터 오류 → 이전 단계들 다시 확인")

    # 메모리 부족 시 자동 해결 시도
    if "out of memory" in str(e).lower():
        print(f"\n🔧 자동 해결 시도: 배치 크기를 1로 줄여서 재시도")
        training_args.per_device_train_batch_size = 1
        training_args.per_device_eval_batch_size = 1

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=ds["train"],
            eval_dataset=ds["test"],
            compute_metrics=compute_metrics,
        )

        try:
            trainer.train()
            print("✅ 배치 크기 1로 훈련 성공!")
        except:
            print("❌ 배치 크기 1로도 실패. 런타임을 재시작하세요.")

print(f"\n✨ 5단계 완료! 차선 검출 모델 훈련이 끝났습니다!")

# 다음 단계 안내
print(f"\n🎯 훈련 완료 후 할 수 있는 것들:")
print(f"  1. 새로운 이미지로 테스트")
print(f"  2. 모델 성능 분석")
print(f"  3. 하이퍼파라미터 튜닝으로 성능 개선")
print(f"  4. 더 많은 데이터로 재훈련")

훈련된 모델로 1.mp4 동영상에서 차선을 검출

In [ ]:
# 동영상 차선 검출 추론
# 훈련된 SegFormer 모델로 1.mp4에서 차선 검출

# 필요한 라이브러리 설치
!pip install -q opencv-python matplotlib

import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import os

print("라이브러리 로드 완료!")

# 🔍 1. 훈련된 모델 로드
print("\n🤖 훈련된 모델 로드 중...")

# 클래스 정의 (훈련할 때와 동일)
id2label = {0: "background", 1: "lane"}
label2id = {v: k for k, v in id2label.items()}

# 모델 경로 확인
model_path = "./segformer-lane-detection"
if os.path.exists(model_path):
    print(f"✅ 모델 폴더 발견: {model_path}")

    # 훈련된 모델 로드
    model = SegformerForSemanticSegmentation.from_pretrained(
        model_path,
        id2label=id2label,
        label2id=label2id
    )

    # 이미지 전처리기 로드
    image_processor = SegformerImageProcessor.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512",
        id2label=id2label,
        label2id=label2id,
        do_reduce_labels=False
    )

    print("✅ 훈련된 모델 로드 완료!")

else:
    print(f"❌ 훈련된 모델이 없습니다: {model_path}")
    print("먼저 모델 훈련을 완료하세요!")
    exit()

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"🔧 추론 장치: {device}")

# 🎥 2. 동영상 파일 확인
video_path = "/content/sample_data/1.mp4"
if not os.path.exists(video_path):
    print(f"❌ 동영상 파일이 없습니다: {video_path}")
    print("1.mp4 파일을 업로드하거나 경로를 확인하세요!")
    exit()

print(f"✅ 동영상 파일 발견: {video_path}")

# 🔧 3. 추론 함수 정의
def detect_lanes_in_frame(frame):
    """단일 프레임에서 차선 검출"""
    # PIL 이미지로 변환
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # 전처리
    inputs = image_processor(pil_image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 추론
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 후처리: 원본 크기로 리사이즈
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=(frame.shape[0], frame.shape[1]),
        mode="bilinear",
        align_corners=False,
    )

    # 예측 마스크 생성
    pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

    return pred_mask

def overlay_mask_on_frame(frame, mask, alpha=0.5):
    """원본 프레임에 차선 마스크 오버레이"""
    # 차선 마스크를 컬러로 변환 (빨간색)
    color_mask = np.zeros_like(frame)
    color_mask[mask == 1] = [0, 0, 255]  # 빨간색으로 차선 표시

    # 원본과 마스크 블렌딩
    result = cv2.addWeighted(frame, 1-alpha, color_mask, alpha, 0)

    return result

# 🎬 4. 동영상 처리 시작
print(f"\n🎬 동영상 처리 시작...")

# 동영상 읽기
cap = cv2.VideoCapture(video_path)

# 동영상 정보 가져오기
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"📊 동영상 정보:")
print(f"  - 해상도: {width}x{height}")
print(f"  - FPS: {fps}")
print(f"  - 총 프레임: {total_frames}")
print(f"  - 예상 처리 시간: 약 {total_frames * 2 / 60:.1f}분")

# 결과 동영상 저장 설정
output_path = "1_lane_detected.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# 프레임별 처리
frame_count = 0
print(f"\n🔄 프레임 처리 중...")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # 차선 검출
        mask = detect_lanes_in_frame(frame)

        # 결과 오버레이
        result_frame = overlay_mask_on_frame(frame, mask, alpha=0.3)

        # 프레임에 정보 텍스트 추가
        cv2.putText(result_frame, f"Frame: {frame_count}/{total_frames}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.putText(result_frame, "Lane Detection",
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # 결과 저장
        out.write(result_frame)

        # 진행 상황 출력 (10프레임마다)
        if frame_count % 10 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"  진행률: {progress:.1f}% ({frame_count}/{total_frames})")

except Exception as e:
    print(f"❌ 처리 중 오류: {e}")

finally:
    # 리소스 정리
    cap.release()
    out.release()
    cv2.destroyAllWindows()

print(f"\n🎉 동영상 처리 완료!")
print(f"📁 결과 파일: {output_path}")

# 🔍 5. 결과 샘플 프레임 시각화
print(f"\n🖼️ 샘플 결과 시각화...")

# 결과 동영상에서 몇 프레임 추출해서 보여주기
cap_result = cv2.VideoCapture(output_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sample_frames = [total_frames//4, total_frames//2, total_frames*3//4]

for i, frame_idx in enumerate(sample_frames):
    cap_result.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap_result.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f"Frame {frame_idx}")
        axes[i].axis('off')

cap_result.release()
plt.tight_layout()
plt.savefig("lane_detection_samples.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 샘플 이미지 저장: lane_detection_samples.png")

# 📊 6. 처리 결과 요약
print(f"\n📊 처리 결과 요약:")
print(f"  - 입력 동영상: {video_path}")
print(f"  - 출력 동영상: {output_path}")
print(f"  - 처리된 프레임: {frame_count}개")
print(f"  - 차선 검출 모델: SegFormer")
print(f"  - 검출된 클래스: 배경, 차선")

# 💡 사용 팁
print(f"\n💡 사용 팁:")
print(f"  - 차선이 빨간색으로 표시됩니다")
print(f"  - 투명도 조정: overlay_mask_on_frame의 alpha 값 변경")
print(f"  - 색상 변경: color_mask[mask == 1] = [B, G, R] 값 변경")
print(f"  - 더 빠른 처리: 프레임 건너뛰기 또는 해상도 축소")

print(f"\n🎯 동영상 차선 검출 완료!")

라벨 확인 코드
클래스를 알 수 있다

In [ ]:
# 1. 실제 JSON에서 모든 라벨 종류 확인
import json
import os
from collections import Counter

label_dir = '/content/sample_data/labels'
json_files = [f for f in os.listdir(label_dir) if f.endswith('.json')]

print("=== 전체 데이터셋의 모든 라벨 분석 ===")
all_labels = Counter()

# 모든 JSON 파일에서 라벨 수집
for json_file in json_files:
    with open(os.path.join(label_dir, json_file), 'r') as f:
        data = json.load(f)

    for obj in data.get('objects', []):
        label = obj.get('label', '')
        all_labels[label] += 1

print(f"발견된 총 라벨 종류: {len(all_labels)}개")
print("\n라벨별 개수:")
for label, count in all_labels.most_common():
    print(f"  {label}: {count}개")

# 2. 차선/차량 관련 라벨들 식별
lane_labels = []
vehicle_labels = []
person_labels = []

for label in all_labels.keys():
    label_lower = label.lower()
    if any(keyword in label_lower for keyword in ['line', 'lane', 'marking']):
        lane_labels.append(label)
    elif any(keyword in label_lower for keyword in ['vehicle', 'car', 'truck', 'bus']):
        vehicle_labels.append(label)
    elif any(keyword in label_lower for keyword in ['person', 'pedestrian', 'human']):
        person_labels.append(label)

print(f"\n=== 분류 결과 ===")
print(f"차선 관련 라벨 ({len(lane_labels)}개): {lane_labels}")
print(f"차량 관련 라벨 ({len(vehicle_labels)}개): {vehicle_labels}")
print(f"사람 관련 라벨 ({len(person_labels)}개): {person_labels}")


🔍 분석 코드 완성!
이 코드를 실행하면:

위치별 차선 분포 확인 (왼쪽 vs 오른쪽)
차선 타입별 크기 분석
불균형 정도 진단
해결책 제시

In [ ]:
# 차선 위치별 분석
# 왜 오른쪽 차선만 잘 인식되는지 분석

import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("🔍 차선 위치별 분석 시작!")

LABEL_DIR = '/content/sample_data/labels'
IMAGE_DIR = '/content/sample_data/JPEGImages_mosaic'

# 차선 라벨들
lane_labels = ['whdot', 'whsol', 'yesol', 'yedot', 'blsol', 'bldot']

# 분석 결과 저장
lane_position_stats = {
    'left_side': Counter(),    # 이미지 왼쪽 절반
    'right_side': Counter(),   # 이미지 오른쪽 절반
    'center': Counter(),       # 이미지 중앙
}

lane_size_stats = {label: [] for label in lane_labels}

print("📊 샘플 이미지들 분석 중...")

# 샘플 JSON 파일들 분석 (시간 절약을 위해 50개만)
json_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')][:50]

for i, json_file in enumerate(json_files):
    if i % 10 == 0:
        print(f"  진행: {i+1}/{len(json_files)}")

    json_path = os.path.join(LABEL_DIR, json_file)

    # 해당 이미지 크기 확인
    image_name = json_file.replace('_gtFine_polygons.json', '_leftImg8bit.png')
    image_path = os.path.join(IMAGE_DIR, image_name)

    if not os.path.exists(image_path):
        continue

    image = cv2.imread(image_path)
    if image is None:
        continue

    height, width = image.shape[:2]

    # JSON 파일 읽기
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except:
        continue

    # 각 차선 객체 분석
    for obj in data.get('objects', []):
        label = obj.get('label', '')

        if label in lane_labels:
            points = np.array(obj['polygon'])

            # 폴리곤의 중심점 계산
            center_x = np.mean(points[:, 0])
            center_y = np.mean(points[:, 1])

            # 폴리곤 크기 계산 (면적)
            area = cv2.contourArea(points.astype(np.int32))
            lane_size_stats[label].append(area)

            # 위치 분류
            if center_x < width * 0.3:  # 왼쪽 30%
                lane_position_stats['left_side'][label] += 1
            elif center_x > width * 0.7:  # 오른쪽 30%
                lane_position_stats['right_side'][label] += 1
            else:  # 중앙 40%
                lane_position_stats['center'][label] += 1

print("✅ 분석 완료!")

# 📊 결과 출력
print(f"\n📊 차선 위치별 분포 (샘플 {len(json_files)}개 기준):")

total_by_position = {}
for position in ['left_side', 'center', 'right_side']:
    total = sum(lane_position_stats[position].values())
    total_by_position[position] = total

    print(f"\n🔍 {position.upper()}:")
    for label in lane_labels:
        count = lane_position_stats[position][label]
        if count > 0:
            print(f"  - {label}: {count}개")

print(f"\n📈 위치별 총합:")
for position, total in total_by_position.items():
    percentage = (total / sum(total_by_position.values())) * 100
    print(f"  - {position}: {total}개 ({percentage:.1f}%)")

# 🎯 차선 타입별 평균 크기 분석
print(f"\n📏 차선 타입별 평균 크기 (픽셀 면적):")
for label in lane_labels:
    if lane_size_stats[label]:
        avg_size = np.mean(lane_size_stats[label])
        print(f"  - {label}: {avg_size:.0f} 픽셀²")

# 🔍 불균형 진단
print(f"\n🎯 불균형 진단:")

# 좌우 불균형 체크
left_total = total_by_position['left_side']
right_total = total_by_position['right_side']
if right_total > left_total * 1.5:
    print(f"❌ 오른쪽 편향: 오른쪽({right_total}) vs 왼쪽({left_total})")
    print(f"   해결책: 이미지 좌우 반전 증강 필요")
elif left_total > right_total * 1.5:
    print(f"❌ 왼쪽 편향: 왼쪽({left_total}) vs 오른쪽({right_total})")
else:
    print(f"✅ 좌우 균형: 왼쪽({left_total}) vs 오른쪽({right_total})")

# 차선 타입 불균형 체크
print(f"\n차선 타입별 편향:")
total_lanes = sum(sum(stats.values()) for stats in lane_position_stats.values())
for label in lane_labels:
    label_total = sum(stats[label] for stats in lane_position_stats.values())
    if label_total > 0:
        percentage = (label_total / total_lanes) * 100
        if percentage > 50:
            print(f"❌ {label} 과다: {percentage:.1f}%")
        elif percentage < 5:
            print(f"⚠️ {label} 부족: {percentage:.1f}%")
        else:
            print(f"✅ {label} 적당: {percentage:.1f}%")

# 💡 해결책 제시
print(f"\n💡 오른쪽 편향 해결책:")
print(f"1. 🔄 데이터 증강: 이미지 좌우 반전 추가")
print(f"2. ⚖️ 위치별 가중치: 왼쪽 차선에 더 높은 가중치")
print(f"3. 🎯 타겟 샘플링: 왼쪽 차선 많은 이미지 선별")
print(f"4. 📏 멀티스케일: 작은 차선도 잘 잡도록 설정")

# 🔧 개선된 데이터 증강 코드 제안
print(f"\n🔧 제안 코드 (데이터 증강에 추가):")
print(f"""
from torchvision.transforms import RandomHorizontalFlip

# 훈련 시 50% 확률로 좌우 반전
def train_transforms_improved(example_batch):
    images = []
    labels = []

    for img, label in zip(example_batch['image'], example_batch['label']):
        # 50% 확률로 좌우 반전
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            label = label.transpose(Image.FLIP_LEFT_RIGHT)

        # 기존 증강 적용
        img = jitter(img.convert("RGB"))
        images.append(img)
        labels.append(label)

    return image_processor(images, labels, return_tensors="pt")
""")

print(f"\n✨ 차선 위치 분석 완료!")
print(f"이 결과를 바탕으로 훈련을 개선해보세요! 🎯")